# 🎓 Examen Final — Pipeline ELT Complet (3 Heures)

**Contexte métier :**
Vous avez été recruté(e) en tant que Data Engineer pour *TheLook eCommerce*.
Votre mission d'aujourd'hui est de construire le pipeline de données quotidien des produits du catalogue.
Ce pipeline extraira les données depuis un fichier stocké dans le cloud (qui remplace une API), les nettoiera, puis calculera des indicateurs clés (KPIs), le tout automatisé via GitHub Actions.

---

### 📋 Règles de l'examen
- Vous avez **3 heures**.
- Le travail s'effectue individuellement.
- Vous devez construire une architecture **Médaillon (Bronze, Silver, Gold)**.
- Toutes les tables BigQuery doivent être créées dans votre dataset personnel : `thelook_<votre_prenom>`.
- Le rendu final sera un dépôt GitHub contenant votre code.

---

## 🛠️ Outils à disposition

| Couche | Outil | Action |
|:---|:---|:---|
| **Source ➔ Bronze** | **Python (dlt)** | Ingestion des données brutes avec `pipeline.run()` |
| **Bronze ➔ Silver** | **SQL (BigQuery)** | Nettoyage, typage (`SAFE_CAST`, `COALESCE`, `NULLIF`) |
| **Silver ➔ Gold** | **SQL (BigQuery)** | Agrégation (`GROUP BY`, `COUNT`, `AVG`) |
| **Orchestration** | **GitHub Actions** | Automatisation CI/CD (Workflow YAML) |

---
## 📦 Étape 1 : Ingestion des données (Couche Bronze) - 30% des points

Le fichier source est un fichier JSON volontairement contenant des erreurs de typage et des données manquantes.
Il est accessible publiquement à cette adresse :
`https://storage.googleapis.com/lasalle-public-data/messy_products.json`

*(Note : vous pouvez copier/coller cette URL dans votre navigateur pour voir à quoi ressemblent les données).*

**Votre mission :**
1. Sur votre machine locale (ou Cloud Shell), créez un dossier de projet et un fichier `load_products.py`.
2. Complétez le script Python ci-dessous (le "Starter Kit") pour ingérer ce fichier JSON dans une table `bronze_products` dans BigQuery.

> 💡 **Astuce d'authentification locale :** Avant de lancer le script Python sur votre terminal, assurez-vous d'être connecté à GCP en tapant :
> `gcloud auth application-default login --project=VOTRE_ID_PROJET`

In [ ]:
# Fichier: ingestion/load_products.py
# --- Starter Kit à copier dans votre projet ---

import dlt
import requests

DATA_URL = "https://storage.googleapis.com/lasalle-public-data/messy_products.json"

def get_messy_products():
    """Récupère les données JSON depuis l'URL"""
    response = requests.get(DATA_URL)
    return response.json()

# 1. Initialisation du pipeline dlt
pipeline = dlt.pipeline(
    pipeline_name="exam_pipeline",
    destination="___",          # REMPLACER : Quel est notre entrepôt de données ?
    dataset_name="___"          # REMPLACER : thelook_<votre_prenom>
)

# 2. Exécution du chargement
load_info = pipeline.run(
    get_messy_products(), 
    table_name="___",           # REMPLACER : Nom de la table cible (ex: bronze_products)
    write_disposition="replace" 
)

print("✅ Chargement terminé !")
print(load_info)


---
## 🧹 Étape 2 : Nettoyage des données (Couche Silver) - 40% des points

Félicitations, vos données sont dans BigQuery ! Mais elles sont très "sales" (Bronze).
Regardez le schéma de votre table `bronze_products` dans la console BigQuery : tout est en format texte (`STRING`), il y a des valeurs nulles, et des symboles `$` dans les prix.

**Votre mission :**
Écrivez une requête SQL (sauvegardée dans un fichier `transformations/silver.sql`) qui lit la table `bronze_products` et crée (ou remplace) une table `silver_products` totalement propre.

**Règles de nettoyage attendues :**
1. `product_id` doit être un entier (`INT64`).
2. `category` ne doit pas contenir de vraies valeurs nulles (remplacez-les par `'Unknown'`) et mettez tout en **majuscules** pour harmoniser.
3. `retail_price` doit être un nombre décimal (`FLOAT64`). Il faudra enlever les symboles `$` ou les textes `"NA"`.
4. `cost` doit être un nombre décimal (`FLOAT64`). Attention à la chaîne de texte `"NULL"`.
5. `is_active` doit être transformé en un vrai booléen (`BOOL`). Si le texte est `"true"`, alors `TRUE`, sinon `FALSE`.

> 💡 **Aide-mémoire SQL :**
> - Pour remplacer une valeur nulle : `COALESCE(colonne, 'valeur_par_defaut')`
> - Pour changer le type d'une colonne de manière sécurisée : `SAFE_CAST(colonne AS FLOAT64)`
> - Pour transformer un mot exact en `null` : `NULLIF(colonne, 'NULL')`
> - Pour une condition spécifique : `CASE WHEN colonne = 'true' THEN TRUE ELSE FALSE END`
> - Pour la création de la table : `CREATE OR REPLACE TABLE \`votre_projet.votre_dataset.silver_products\` AS SELECT ...`

In [ ]:
-- Fichier: transformations/silver.sql
-- Écrivez votre requête de transformation ici :

CREATE OR REPLACE TABLE `votre_projet.thelook_votre_prenom.silver_products` AS
SELECT
    -- À COMPLÉTER
FROM
    `votre_projet.thelook_votre_prenom.bronze_products`;


---
## 📈 Étape 3 : Agrégation Métier (Couche Gold) - 15% des points

Maintenant que vous avez une belle table `silver_products` avec des types corrects, nous voulons extraire de la valeur métier.

**Votre mission :**
Écrivez une requête SQL (sauvegardée dans `transformations/gold.sql`) qui crée (ou remplace) une table `gold_category_metrics`.

**Résultat attendu :**
Pour chaque **catégorie** de vêtement, nous voulons connaître :
1. Le nombre total de produits (`total_products`).
2. Le prix de vente moyen (`avg_retail_price`).
3. La marge unitaire moyenne (`avg_margin` = moyenne de `retail_price - cost`).

Triez le résultat pour afficher les catégories les plus rentables (marge moyenne la plus élevée) en premier.

In [ ]:
-- Fichier: transformations/gold.sql
-- Écrivez votre requête d'agrégation ici :

CREATE OR REPLACE TABLE `votre_projet.thelook_votre_prenom.gold_category_metrics` AS
SELECT
    category,
    -- À COMPLÉTER
FROM
    `votre_projet.thelook_votre_prenom.silver_products`
GROUP BY
    category
ORDER BY
    avg_margin DESC;


---
## ⚙️ Étape 4 : Orchestration (CI/CD GitHub Actions) - 15% des points

Vous avez 3 scripts (`load_products.py`, `silver.sql`, `gold.sql`). En production, on ne va pas les lancer à la main tous les jours ! Nous allons utiliser GitHub Actions pour orchestrer l'ordre d'exécution (le pipeline ELT).

**Votre mission :**
1. Sur votre dépôt GitHub, créez le dossier `.github/workflows/` et ajoutez-y le fichier `elt_pipeline.yml` ci-dessous.
2. Décommentez et complétez les 3 étapes (Jobs) à la fin du fichier.
3. *Le professeur a déjà configuré une Workload Identity Federation (WIF) dans Google Cloud. Vous n'avez pas besoin de clé JSON !* Suivez simplement les instructions d'authentification ci-dessous.

> 💡 **Exécution des requêtes SQL depuis le terminal :** 
> `bq query --use_legacy_sql=false < transformations/votre_fichier.sql`

In [ ]:
# Fichier: .github/workflows/elt_pipeline.yml
name: Pipeline ELT Quotidien

# Le pipeline se lance manuellement, ou tous les jours à minuit
on:
  workflow_dispatch:
  schedule:
    - cron: '0 0 * * *'

jobs:
  run-elt:
    runs-on: ubuntu-latest
    
    # Permissions requises pour utiliser Workload Identity Federation (OIDC) avec GitHub Actions
    permissions:
      contents: 'read'
      id-token: 'write'

    steps:
      - name: 📥 Récupérer le code (Checkout)
        uses: actions/checkout@v4

      - name: 🔐 Authentification Google Cloud (WIF)
        uses: google-github-actions/auth@v2
        with:
          # (Le formateur fournira ces identifiants exacts le jour de l'examen)
          workload_identity_provider: 'projects/VOTRE_NUMERO_PROJET/locations/global/workloadIdentityPools/github-pool/providers/github-provider'
          service_account: 'votre-service-account@VOTRE_PROJET.iam.gserviceaccount.com'

      - name: 🐍 Installation de Python et dlt
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      - run: |
          pip install dlt[bigquery] requests

      # --- 🟢 ÉTAPE BRONZE : INGESTION ---
      # - name: 1️⃣ Ingestion API vers Bronze
      #   run: # À COMPLÉTER (Quelle commande taper pour exécuter le script Python ?)

      # --- ⚪ ÉTAPE SILVER : TRANSFORMATION ---
      # - name: 2️⃣ Transformation Bronze vers Silver
      #   run: # À COMPLÉTER (Utiliser bq query avec redirection d'entrée < ...)

      # --- 🟡 ÉTAPE GOLD : AGRÉGATION ---
      # - name: 3️⃣ Agrégation Silver vers Gold
      #   run: # À COMPLÉTER


---
### ✅ Livrable attendu

Sur votre repo GitHub, la structure de fichiers doit ressembler exactement à ceci :

```
mon_projet_exam/
│
├── .github/
│   └── workflows/
│       └── elt_pipeline.yml
│
├── ingestion/
│   └── load_products.py
│
└── transformations/
    ├── silver.sql
    └── gold.sql
```

**Bon courage !**